## Initial Observations

### Dataset Scale
The Lending Club dataset is genuinely large‑scale: approximately **2.26 million rows × 151 columns**, with the raw CSV occupying **~1.55 GB**. Even selective column loading consumes over **220 MB**, confirming that full‑file operations require careful memory management and incremental processing.

### Loan Status Structure
The loan status distribution shows a clear pattern: **47.6% Fully Paid**, **38.8% Current**, and **11.9% Charged Off**, with the remaining statuses making up small fractions. When focusing only on terminal loans (Fully Paid + Charged Off), the effective class balance is close to **80/20**, which is workable for modelling default risk.

### Data Challenges
Several structural issues are immediately visible:
- Full‑file loads are **memory‑constrained**, making column selection essential.
- `low_memory=False` is **not feasible** at this scale.
- The `id` column contains **mixed types**, triggering dtype warnings.
- Many columns are **conditional or sparsely populated**, especially those related to joint applications, secondary applicants, or optional borrower disclosures.

These challenges imply that preprocessing will require careful dtype handling, null‑management, and selective feature inclusion.

### Analytical Questions Raised
These observations naturally lead to the core analytical questions for the capstone:
- **How well does the Lending Club Grade predict default rate?**
- **Do defaults cluster by loan purpose, state, or income tier?**
- **Has the default rate changed over the 11‑year lending window?**
- **Which borrower characteristics correlate with default beyond what the Grade captures?**

This section forms the foundation for scoping the capstone analysis over the coming days.


In [3]:
# Step 6 - Confirm Scale of Dataset.

scale_check = pd.read_csv(
    csv_path,
    usecols=["id", "loan_amnt", "loan_status"]
)
print(f"Full dataset shape: {scale_check.shape}")
print(f"Memory used: {scale_check.memory_usage(deep=True).sum() / (1024**2):.2f} MB")
print()
print("Loan status distribution:")
print(scale_check["loan_status"].value_counts())



C:\Users\Mark PC\AppData\Local\Temp\ipykernel_17816\4274886038.py:3: DtypeWarning: Columns (0: id) have mixed types. Specify dtype option on import or set low_memory=False.
  scale_check = pd.read_csv(


Full dataset shape: (2260701, 3)
Memory used: 221.46 MB

Loan status distribution:
loan_status
Fully Paid                                             1076751
Current                                                 878317
Charged Off                                             268559
Late (31-120 days)                                       21467
In Grace Period                                           8436
Late (16-30 days)                                         4349
Does not meet the credit policy. Status:Fully Paid        1988
Does not meet the credit policy. Status:Charged Off        761
Default                                                     40
Name: count, dtype: int64


## Step 5 - Analytical Questions Worth Flagging.

- Q1. Which columns look genuinely useful for default prediction analysis?
- Q2. Which columns look like noise or too-many-nulls to be useful?
- Q3. What immediate analytical questions come to mind? (2-3 sentences)
- Q4. What's confusing or unclear? (things worth researching later)

In [10]:
# Step 4 - Initial Data Inspection.

df_sample.head()
df_sample.dtypes.value_counts()
df_sample.isnull().sum().sort_values(ascending=False).head(20)

member_id                                     10000
sec_app_num_rev_accts                         10000
sec_app_open_act_il                           10000
sec_app_inq_last_6mths                        10000
sec_app_open_acc                              10000
sec_app_mort_acc                              10000
sec_app_mths_since_last_major_derog           10000
sec_app_collections_12_mths_ex_med            10000
sec_app_chargeoff_within_12_mths              10000
sec_app_fico_range_low                        10000
sec_app_earliest_cr_line                      10000
sec_app_revol_util                            10000
sec_app_fico_range_high                       10000
revol_bal_joint                               10000
desc                                           9999
verification_status_joint                      9935
dti_joint                                      9935
annual_inc_joint                               9935
orig_projected_additional_accrued_interest     9934
payment_plan

In [9]:
# Step 3 - Load a Sample from Dataset.

# Load only the first 10,000 rows for exploration
df_sample = pd.read_csv(csv_path, nrows=10000, low_memory=False)

print(f"Sample shape: {df_sample.shape}")
print(f"Memory used: {df_sample.memory_usage(deep=True).sum() / (1024**2):.2f} MB")

Sample shape: (10000, 151)
Memory used: 25.61 MB


In [8]:
# Step 2 - Look at Column Names w/o loading Data.

# Read just the first row to get column names
columns_only = pd.read_csv(csv_path, nrows=0)
print(f"Number of columns: {len(columns_only.columns)}")
print()
print("Column names:")
for col in columns_only.columns:
    print(f"  - {col}")

Number of columns: 151

Column names:
  - id
  - member_id
  - loan_amnt
  - funded_amnt
  - funded_amnt_inv
  - term
  - int_rate
  - installment
  - grade
  - sub_grade
  - emp_title
  - emp_length
  - home_ownership
  - annual_inc
  - verification_status
  - issue_d
  - loan_status
  - pymnt_plan
  - url
  - desc
  - purpose
  - title
  - zip_code
  - addr_state
  - dti
  - delinq_2yrs
  - earliest_cr_line
  - fico_range_low
  - fico_range_high
  - inq_last_6mths
  - mths_since_last_delinq
  - mths_since_last_record
  - open_acc
  - pub_rec
  - revol_bal
  - revol_util
  - total_acc
  - initial_list_status
  - out_prncp
  - out_prncp_inv
  - total_pymnt
  - total_pymnt_inv
  - total_rec_prncp
  - total_rec_int
  - total_rec_late_fee
  - recoveries
  - collection_recovery_fee
  - last_pymnt_d
  - last_pymnt_amnt
  - next_pymnt_d
  - last_credit_pull_d
  - last_fico_range_high
  - last_fico_range_low
  - collections_12_mths_ex_med
  - mths_since_last_major_derog
  - policy_code
  - ap

In [2]:
# Step 1 - Initial Setup & Verification of Dataset.
import pandas as pd
from pathlib import Path

import os

# Path to the CSV
csv_path = Path("../data/accepted_2007_to_2018Q4.csv")

# Verify it exists
print(f"File exists: {csv_path.exists()}")
print(f"File size: {csv_path.stat().st_size / (1024**3):.2f} GB")

File exists: True
File size: 1.56 GB
